In [1]:
import os
from urllib.request import urlretrieve

import pandas as pd
import snowflake.connector

In [2]:
ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
ZONE_LOOKUP_LOCAL = "../data/taxi_zone_lookup.csv"

DATABASE = os.getenv("SNOWFLAKE_DATABASE")
RAW_SCHEMA = os.getenv("SNOWFLAKE_SCHEMA_RAW")
ANALYTICS_SCHEMA = os.getenv("SNOWFLAKE_SCHEMA_ANALYTICS")

ZONE_TABLE = "TAXI_ZONE_LOOKUP"
STAGE_TABLE = "TRIPS_ENRICHED_UNIFIED_STAGE"

In [3]:
conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=DATABASE,
    role=os.getenv("SNOWFLAKE_ROLE"),
)

cur = conn.cursor()

cur.execute(f"USE DATABASE {DATABASE}")
cur.execute(f"CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA}")
cur.execute(f"CREATE SCHEMA IF NOT EXISTS {ANALYTICS_SCHEMA}")

print("Snowflake connection ready")

Snowflake connection ready


In [4]:
os.makedirs(os.path.dirname(ZONE_LOOKUP_LOCAL), exist_ok=True)

if not os.path.exists(ZONE_LOOKUP_LOCAL):
    urlretrieve(ZONE_LOOKUP_URL, ZONE_LOOKUP_LOCAL)

print("Taxi Zone Lookup ready:", ZONE_LOOKUP_LOCAL)

Taxi Zone Lookup ready: ../data/taxi_zone_lookup.csv


In [5]:
cur.execute(f"""
CREATE OR REPLACE TABLE {ANALYTICS_SCHEMA}.{ZONE_TABLE} (
    location_id INTEGER,
    borough STRING,
    zone STRING
)
""")

print("Zone lookup table ready")

Zone lookup table ready


In [6]:
from snowflake.connector.pandas_tools import write_pandas

zones_df = pd.read_csv(ZONE_LOOKUP_LOCAL)
zones_df = zones_df.rename(columns={
    "LocationID": "location_id",
    "Borough": "borough",
    "Zone": "zone"
})[["location_id", "borough", "zone"]]

success, nchunks, nrows, _ = write_pandas(
    conn,
    zones_df,
    ZONE_TABLE,
    database=DATABASE,
    schema=ANALYTICS_SCHEMA,
    auto_create_table=False,
    overwrite=True,
    quote_identifiers=False,
    use_logical_type=True
)

print("Lookup load success:", success)
print("Rows written:", nrows)

Lookup load success: True
Rows written: 265


In [7]:
cur.execute(f"""
CREATE OR REPLACE TABLE {ANALYTICS_SCHEMA}.{STAGE_TABLE} (
    vendor_id INTEGER,
    vendor_name STRING,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    passenger_count FLOAT,
    trip_distance FLOAT,
    rate_code_id INTEGER,
    rate_code_desc STRING,
    store_and_fwd_flag STRING,
    pu_location_id INTEGER,
    pu_zone STRING,
    pu_borough STRING,
    do_location_id INTEGER,
    do_zone STRING,
    do_borough STRING,
    payment_type INTEGER,
    payment_type_desc STRING,
    trip_type INTEGER,
    fare_amount FLOAT,
    extra FLOAT,
    mta_tax FLOAT,
    tip_amount FLOAT,
    tolls_amount FLOAT,
    ehail_fee FLOAT,
    improvement_surcharge FLOAT,
    congestion_surcharge FLOAT,
    airport_fee FLOAT,
    total_amount FLOAT,
    run_id STRING,
    source_year INTEGER,
    source_month INTEGER,
    service_type STRING,
    ingested_at_utc TIMESTAMP
)
""")

print("Stage table ready")

Stage table ready


In [8]:
cur.execute(f"""
INSERT INTO {ANALYTICS_SCHEMA}.{STAGE_TABLE}
SELECT
    y.vendor_id,
    CASE y.vendor_id
        WHEN 1 THEN 'Creative Mobile Technologies'
        WHEN 2 THEN 'Curb Mobility'
        WHEN 6 THEN 'Myle Technologies'
        WHEN 7 THEN 'Helix'
        ELSE 'Unknown'
    END AS vendor_name,

    y.pickup_datetime,
    y.dropoff_datetime,
    y.passenger_count,
    y.trip_distance,

    CAST(y.rate_code_id AS INTEGER) AS rate_code_id,
    CASE CAST(y.rate_code_id AS INTEGER)
        WHEN 1 THEN 'Standard rate'
        WHEN 2 THEN 'JFK'
        WHEN 3 THEN 'Newark'
        WHEN 4 THEN 'Nassau/Westchester'
        WHEN 5 THEN 'Negotiated fare'
        WHEN 6 THEN 'Group ride'
        ELSE 'Unknown'
    END AS rate_code_desc,

    y.store_and_fwd_flag,

    y.pu_location_id,
    pu.zone AS pu_zone,
    pu.borough AS pu_borough,

    y.do_location_id,
    do.zone AS do_zone,
    do.borough AS do_borough,

    CAST(y.payment_type AS INTEGER) AS payment_type,
    CASE CAST(y.payment_type AS INTEGER)
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
        ELSE 'Unknown'
    END AS payment_type_desc,

    NULL AS trip_type,

    y.fare_amount,
    y.extra,
    y.mta_tax,
    y.tip_amount,
    y.tolls_amount,
    NULL AS ehail_fee,
    y.improvement_surcharge,
    y.congestion_surcharge,
    y.airport_fee,
    y.total_amount,

    y.run_id,
    y.source_year,
    y.source_month,
    y.service_type,
    y.ingested_at_utc

FROM {RAW_SCHEMA}.YELLOW_TRIPS_RAW y
LEFT JOIN {ANALYTICS_SCHEMA}.{ZONE_TABLE} pu
    ON y.pu_location_id = pu.location_id
LEFT JOIN {ANALYTICS_SCHEMA}.{ZONE_TABLE} do
    ON y.do_location_id = do.location_id
""")

print("Yellow enrichment inserted")

Yellow enrichment inserted


In [9]:
cur.execute(f"""
INSERT INTO {ANALYTICS_SCHEMA}.{STAGE_TABLE}
SELECT
    g.vendor_id,
    CASE g.vendor_id
        WHEN 1 THEN 'Creative Mobile Technologies'
        WHEN 2 THEN 'Curb Mobility'
        WHEN 6 THEN 'Myle Technologies'
        WHEN 7 THEN 'Helix'
        ELSE 'Unknown'
    END AS vendor_name,

    g.pickup_datetime,
    g.dropoff_datetime,
    g.passenger_count,
    g.trip_distance,

    CAST(g.rate_code_id AS INTEGER) AS rate_code_id,
    CASE CAST(g.rate_code_id AS INTEGER)
        WHEN 1 THEN 'Standard rate'
        WHEN 2 THEN 'JFK'
        WHEN 3 THEN 'Newark'
        WHEN 4 THEN 'Nassau/Westchester'
        WHEN 5 THEN 'Negotiated fare'
        WHEN 6 THEN 'Group ride'
        ELSE 'Unknown'
    END AS rate_code_desc,

    g.store_and_fwd_flag,

    g.pu_location_id,
    pu.zone AS pu_zone,
    pu.borough AS pu_borough,

    g.do_location_id,
    do.zone AS do_zone,
    do.borough AS do_borough,

    CAST(g.payment_type AS INTEGER) AS payment_type,
    CASE CAST(g.payment_type AS INTEGER)
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
        ELSE 'Unknown'
    END AS payment_type_desc,

    CAST(g.trip_type AS INTEGER) AS trip_type,

    g.fare_amount,
    g.extra,
    g.mta_tax,
    g.tip_amount,
    g.tolls_amount,
    g.ehail_fee,
    g.improvement_surcharge,
    g.congestion_surcharge,
    NULL AS airport_fee,
    g.total_amount,

    g.run_id,
    g.source_year,
    g.source_month,
    g.service_type,
    g.ingested_at_utc

FROM {RAW_SCHEMA}.GREEN_TRIPS_RAW g
LEFT JOIN {ANALYTICS_SCHEMA}.{ZONE_TABLE} pu
    ON g.pu_location_id = pu.location_id
LEFT JOIN {ANALYTICS_SCHEMA}.{ZONE_TABLE} do
    ON g.do_location_id = do.location_id
""")

print("Green enrichment inserted")

Green enrichment inserted


In [10]:
queries = {
    "total_rows": f"""
        SELECT COUNT(*) AS total_rows
        FROM {ANALYTICS_SCHEMA}.{STAGE_TABLE}
    """,
    "rows_by_service": f"""
        SELECT service_type, COUNT(*) AS total_rows
        FROM {ANALYTICS_SCHEMA}.{STAGE_TABLE}
        GROUP BY service_type
        ORDER BY service_type
    """,
    "rows_by_month": f"""
        SELECT service_type, source_year, source_month, COUNT(*) AS total_rows
        FROM {ANALYTICS_SCHEMA}.{STAGE_TABLE}
        GROUP BY service_type, source_year, source_month
        ORDER BY service_type, source_year, source_month
    """,
    "sample_rows": f"""
        SELECT
            service_type,
            vendor_id,
            vendor_name,
            payment_type,
            payment_type_desc,
            rate_code_id,
            rate_code_desc,
            pu_zone,
            pu_borough,
            do_zone,
            do_borough
        FROM {ANALYTICS_SCHEMA}.{STAGE_TABLE}
        LIMIT 20
    """
}

for name, query in queries.items():
    print(f"\n--- {name} ---")
    cur.execute(query)
    rows = cur.fetchmany(20)
    for row in rows:
        print(row)


--- total_rows ---
(869792294,)

--- rows_by_service ---
('green', 68239054)
('yellow', 801553240)

--- rows_by_month ---
('green', 2015, 1, 1508493)
('green', 2015, 2, 1574830)
('green', 2015, 3, 1722574)
('green', 2015, 4, 1664394)
('green', 2015, 5, 1786848)
('green', 2015, 6, 1638868)
('green', 2015, 7, 1541671)
('green', 2015, 8, 1532343)
('green', 2015, 9, 1494927)
('green', 2015, 10, 1630536)
('green', 2015, 11, 1529984)
('green', 2015, 12, 1608297)
('green', 2016, 1, 1445292)
('green', 2016, 2, 1510722)
('green', 2016, 3, 1576393)
('green', 2016, 4, 1543926)
('green', 2016, 5, 1536979)
('green', 2016, 6, 1404727)
('green', 2016, 7, 1332510)
('green', 2016, 8, 1247675)

--- sample_rows ---
('yellow', 2, 'Curb Mobility', 1, 'Credit card', 1, 'Standard rate', 'Penn Station/Madison Sq West', 'Manhattan', 'Midtown East', 'Manhattan')
('yellow', 2, 'Curb Mobility', 1, 'Credit card', 1, 'Standard rate', 'Meatpacking/West Village West', 'Manhattan', 'Midtown South', 'Manhattan')
('yel

In [11]:
cur.execute(f"""
SELECT
    SUM(CASE WHEN pu_zone IS NULL THEN 1 ELSE 0 END) AS null_pu_zone,
    SUM(CASE WHEN pu_borough IS NULL THEN 1 ELSE 0 END) AS null_pu_borough,
    SUM(CASE WHEN do_zone IS NULL THEN 1 ELSE 0 END) AS null_do_zone,
    SUM(CASE WHEN do_borough IS NULL THEN 1 ELSE 0 END) AS null_do_borough
FROM {ANALYTICS_SCHEMA}.{STAGE_TABLE}
""")

print(cur.fetchone())

(9815880, 722902, 9144880, 2457798)


In [12]:
cur.close()
conn.close()
print("Notebook 02 finished")

Notebook 02 finished
